In [1]:
import torch
import clip
from PIL import Image
import os
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

/nix/store/k0d8977kvsg83v7540wsp6hp42w4c1gd-python3-3.13.11-env/lib/python3.13/site-packages/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


cuda


# Stage 1

In [2]:
DATASET_PATH = Path("./assignment4_images/")
assert os.path.isdir(DATASET_PATH), f"Please load the dataset to the following path: {DATASET_PATH}"

images = {
    name: Image.open(DATASET_PATH / name)
    for name in os.listdir(DATASET_PATH)
    if os.path.splitext(name)[-1] == ".jpg"
}

## Stage 2

## Load CLIP Model

In [3]:
clip_model, clip_preproc = clip.load("ViT-B/32", device=device)

## Compute image embeddings

In [4]:
db_embeddings = {
    name: clip_model.encode_image(clip_preproc(img).unsqueeze(0).to(device)).squeeze(0)
    for name, img in images.items()
}

## Retrieval Function

In [5]:
_cos = torch.nn.CosineSimilarity(dim=0)
def clip_retrieve(query: str, top_k: int=5) -> list[tuple[int, float]]:
    with torch.no_grad():
        q_embedding = clip_model.encode_text(clip.tokenize(query).to(device)).squeeze(0)
        similarities = {
            name: _cos(q_embedding, e)
            for name, e in db_embeddings.items()
        }
        return [
            (int(os.path.splitext(name)[0].split("_")[1]), float(score))
            for name, score in sorted(similarities.items(), key=lambda t: t[1])[:top_k]
        ]